<a href="https://colab.research.google.com/github/yilinw762/ds2002-fa26/blob/main/notebooks/04-json-apis/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [8]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item from an order

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [9]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

assert len(lines) == 4
assert lines['qty'].sum() == 7

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [10]:
# TODO
vendors = pd.json_normalize(vendors_json)

joined = lines.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True
)

unmatched = joined[joined['_merge'] == 'left_only']

print("Unmatched vendors:")
print(unmatched[['vendor_id', 'qty']])

print("Units attached to unmatched vendors:", unmatched['qty'].sum())

joined

Unmatched vendors:
Empty DataFrame
Columns: [vendor_id, qty]
Index: []
Units attached to unmatched vendors: 0


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-01,Hoos Burgers,A,both


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [11]:
# TODO
joined['vendor_display'] = joined['name'].fillna(joined['vendor_id'])

vendor_totals = (
    joined.groupby('vendor_display', as_index=False)['qty']
    .sum()
    .rename(columns={
        'vendor_display': 'vendor',
        'qty': 'total_qty'
    })
)

vendor_totals

,vendor,total_qty
0,Cav Merch,1
1,Hoos Burgers,6


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [12]:
# TODO
zone_totals = (
    joined.groupby('zone', dropna=False, as_index=False)['qty']
    .sum()
    .rename(columns={'qty': 'total_qty'})
)

print(zone_totals)

top_zone = zone_totals.loc[zone_totals['total_qty'].idxmax()]

print("Zone with the most items:", top_zone['zone'])
print("Items sold:", top_zone['total_qty'])
print("Total units represented in zone totals:", zone_totals['total_qty'].sum())

missing_zone_units = joined.loc[joined['zone'].isna(), 'qty'].sum()
print("Units with no zone:", missing_zone_units)

  zone  total_qty
0    A          7
Zone with the most items: A
Items sold: 7
Total units represented in zone totals: 7
Units with no zone: 0


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [13]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
orders_with_lines = [
    order for order in ragged_json
    if 'lines' in order
]

ragged_lines = pd.json_normalize(
    orders_with_lines,
    record_path='lines',
    meta=['order', 'vendor_id'],
    errors='ignore'
)

ragged_lines['qty'] = ragged_lines['qty'].astype('Int64')

ragged_lines

,item,qty,order,vendor_id
0,Soda,2,4,V-01
1,Fries,<NA>,6,V-01


### Q6 — Validate

In [14]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** "One row per line item" is useful because each individual item and its quantity can be analyzed directly. For example, calculating the total quantity sold for each vendor becomes easy with a `groupby`. If we kept one row per order, the line items would remain nested inside lists, so we would have to unpack those lists before doing item-level analysis.

**b)** A missing quantity means that the quantity is unknown or was not provided, while a quantity of zero means that we know no units were ordered. They represent different information. If we replace missing quantities with zero, downstream calculations may incorrectly treat unknown data as confirmed zero sales. Keeping the missing value as `<NA>` preserves this distinction.